# ML: Embeddings, Clustering y Agente LLM

Procesa los 68,049 productos sin clasificar usando:
1. **Embeddings**: sentence-transformers multilingual
2. **Clustering**: K-means para agrupar productos similares
3. **Agente LLM**: Databricks Foundation Models para sugerir categorías

## Objetivos:
- Identificar patrones semánticos que las reglas keyword no capturan
- Sugerir categorías para productos ambiguos
- Preparar training data para modelo supervisado futuro

In [0]:
%pip install sentence-transformers scikit-learn --quiet
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, lit, current_timestamp
import json

print("✅ Librerías cargadas exitosamente")

In [0]:
# Extraer productos sin clasificar (id_categoria IS NULL)
productos_sin_cat = spark.sql("""
    SELECT DISTINCT p.id_producto, p.producto
    FROM workspace.superapp.silver_prices p
    LEFT JOIN workspace.superapp.gold_productos_categorias pc 
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NULL
""").toPandas()

print(f"📊 Productos sin clasificar: {len(productos_sin_cat):,}")
print(f"\nMuestra de productos:")
for i, row in productos_sin_cat.head(10).iterrows():
    print(f"  - {row['producto']}")

In [0]:
# Generar embeddings con modelo multilingual
print("🤖 Cargando modelo sentence-transformers...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f"\n📊 Generando embeddings para {len(productos_sin_cat):,} productos...")
embeddings = model.encode(
    productos_sin_cat['producto'].tolist(), 
    show_progress_bar=True,
    batch_size=64
)

print(f"\n✅ Embeddings generados: shape {embeddings.shape}")
productos_sin_cat['embedding'] = list(embeddings)

In [0]:
# Clustering ultra-granular: 1 cluster por cada ~184 productos (antes 400)
# Esto generará ~400 clusters para máxima separación de categorías
n_clusters = max(200, min(500, len(productos_sin_cat) // 184))
print(f"🎯 Aplicando K-means con {n_clusters} clusters...")

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, max_iter=300)
productos_sin_cat['cluster_id'] = kmeans.fit_predict(embeddings)

print(f"\n✅ Clustering completado")
print(f"\nDistribución de clusters (top 10):")
cluster_counts = productos_sin_cat['cluster_id'].value_counts().head(10)
for cluster_id, count in cluster_counts.items():
    print(f"  Cluster {cluster_id}: {count} productos")

In [0]:
# Mostrar samples de los primeros 8 clusters
print("\n" + "="*80)
print("SAMPLES DE CLUSTERS (primeros 8)")
print("="*80)

for cluster_id in range(min(8, n_clusters)):
    cluster_samples = productos_sin_cat[productos_sin_cat['cluster_id'] == cluster_id]
    print(f"\n📦 CLUSTER {cluster_id} ({len(cluster_samples)} productos)")
    print("-" * 80)
    for i, desc in enumerate(cluster_samples['producto'].head(12), 1):
        print(f"  {i:2d}. {desc}")

In [0]:
# Análisis en profundidad de primeros 20 clusters para definir categorías específicas
print("\n" + "="*100)
print("ANÁLISIS DETALLADO: PRIMEROS 20 CLUSTERS")
print("Objetivo: Identificar categorías específicas (manzanas, papel_higienico, asado, etc)")
print("="*100)

for cluster_id in range(min(20, n_clusters)):
    cluster_samples = productos_sin_cat[productos_sin_cat['cluster_id'] == cluster_id]
    
    print(f"\n{'='*100}")
    print(f"📦 CLUSTER {cluster_id} → {len(cluster_samples)} productos")
    print(f"{'='*100}")
    
    # Mostrar 20 productos para ver mejor el patrón
    print("\n🔍 Productos (primeros 20):")
    for i, producto in enumerate(cluster_samples['producto'].head(20), 1):
        print(f"   {i:2d}. {producto}")
    
    # Análisis básico de keywords comunes
    all_text = ' '.join(cluster_samples['producto'].head(50).str.lower().tolist())
    keywords_comunes = []
    
    # Buscar palabras clave frecuentes
    test_keywords = ['agua', 'leche', 'yogur', 'queso', 'pan', 'arroz', 'aceite', 
                     'jabon', 'shampoo', 'detergente', 'limpiador', 'papel', 'vino',
                     'cerveza', 'jugo', 'gaseosa', 'carne', 'pollo', 'chocolate',
                     'alfajor', 'galletitas', 'fideos', 'yerba', 'cafe', 'te',
                     'papa', 'tomate', 'cebolla', 'manzana', 'banana', 'naranja']
    
    for kw in test_keywords:
        if kw in all_text:
            count = all_text.count(kw)
            if count >= 3:  # Al menos 3 menciones
                keywords_comunes.append((kw, count))
    
    if keywords_comunes:
        keywords_comunes.sort(key=lambda x: x[1], reverse=True)
        print(f"\n💡 Keywords frecuentes: {', '.join([f'{kw}({cnt})' for kw, cnt in keywords_comunes[:5]])}")
    
    print(f"\n📝 CATEGORÍA SUGERIDA MANUAL: ___________________")
    print(f"   (Completar después de revisar productos)")
    print()

In [0]:
# Guardar clusters en tabla Delta
spark_df = spark.createDataFrame(
    productos_sin_cat[['id_producto', 'cluster_id']]
)

spark_df.write.mode("overwrite").saveAsTable("workspace.superapp.gold_productos_clusters")

print(f"✅ Clusters guardados en workspace.superapp.gold_productos_clusters")
print(f"   Total registros: {spark_df.count():,}")

## Paso 4: Agente LLM para Tagging

Usa Databricks Foundation Models para sugerir categorías basadas en samples de cada cluster.

**Nota:** La API de Foundation Models requiere autenticación. Si hay errores, las sugerencias se marcarán como 'sin_clasificar'.

In [0]:
# Obtener categorías disponibles
categorias_df = spark.table("workspace.superapp.gold_categorias").filter(col("nivel") == "categoria")
available_categories = [row.nombre for row in categorias_df.select("nombre").collect()]
cat_id_map = {row.nombre: row.id_categoria for row in categorias_df.select("nombre", "id_categoria").collect()}

print(f"🏷️  Categorías disponibles ({len(available_categories)}):")
for i, cat in enumerate(available_categories, 1):
    print(f"  {i:2d}. {cat}")

In [0]:
def tag_cluster_with_llm(cluster_samples, cluster_id, available_categories):
    """
    Usa Databricks Foundation Model para sugerir categoría.
    Retorna el nombre de la categoría sugerida.
    """
    import requests
    
    try:
        # Obtener token y workspace URL
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        
        # Preparar prompt
        samples_text = "\n".join([f"- {s}" for s in cluster_samples[:20]])
        categories_text = ", ".join(available_categories)
        
        prompt = f"""Analiza estos productos de supermercado argentino y sugiere UNA categoría:

Productos:
{samples_text}

Categorías disponibles:
{categories_text}

Responde SOLO con el nombre exacto de la categoría (una palabra). Si ninguna aplica, responde "sin_clasificar"."""

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        data = {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 20,
            "temperature": 0.1
        }
        
        # Endpoint actualizado: Meta Llama 3.3 70B Instruct
        response = requests.post(
            f"https://{workspace_url}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations",
            headers=headers,
            json=data,
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            suggested_cat = result['choices'][0]['message']['content'].strip().lower()
            # Validar que está en la lista
            if suggested_cat in available_categories:
                return suggested_cat
            else:
                # Buscar match parcial
                for cat in available_categories:
                    if cat in suggested_cat or suggested_cat in cat:
                        return cat
                return "sin_clasificar"
        else:
            print(f"  ⚠️  Error HTTP {response.status_code}")
            return "sin_clasificar"
            
    except Exception as e:
        print(f"  ⚠️  Error: {str(e)[:100]}")
        return "sin_clasificar"

print("✅ Función de tagging LLM definida")

In [0]:
print(f"🤖 Tagging de primeros 30 clusters (antes 15)...\n")

# Obtener categorías disponibles
available_categories = [row['nombre'] for row in spark.table("workspace.superapp.gold_categorias").select('nombre').collect()]

suggestions = []

for cluster_id in range(min(30, n_clusters)):
    cluster_products = productos_sin_cat[productos_sin_cat['cluster_id'] == cluster_id]['producto'].tolist()
    
    if len(cluster_products) == 0:
        continue
    
    print(f"Cluster {cluster_id} ({len(cluster_products)} productos):")
    print(f"  Muestras: {cluster_products[:5]}")
    
    # Llamar al LLM
    suggested_category = tag_cluster_with_llm(cluster_products[:20], cluster_id, available_categories)
    print(f"  ✅ Sugerencia: {suggested_category}\n")
    
    suggestions.append({
        'cluster_id': cluster_id,
        'cluster_size': len(cluster_products),
        'sample_products': str(cluster_products[:10]),
        'suggested_category': suggested_category,
        'confidence': 0.6,
        'review_required': suggested_category == 'sin_clasificar'
    })

# Guardar sugerencias con overwriteSchema para permitir cambio de schema
suggestions_df = spark.createDataFrame(suggestions)
suggestions_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.superapp.gold_cluster_suggestions')

print(f"\n✅ Sugerencias guardadas en gold_cluster_suggestions ({len(suggestions)} clusters)")

In [0]:
# Convertir a DataFrame y guardar
if cluster_suggestions:
    suggestions_df = spark.createDataFrame(cluster_suggestions)
    suggestions_df.write.mode("overwrite").saveAsTable("workspace.superapp.gold_cluster_suggestions")
    
    print(f"✅ Sugerencias guardadas en workspace.superapp.gold_cluster_suggestions")
    print(f"   Total clusters procesados: {len(cluster_suggestions)}")
    
    # Mostrar resumen
    print("\n📊 Resumen de sugerencias:")
    suggestions_summary = spark.table("workspace.superapp.gold_cluster_suggestions") \
        .groupBy("suggested_category") \
        .count() \
        .orderBy(col("count").desc())
    suggestions_summary.show(truncate=False)
else:
    print("⚠️  No se generaron sugerencias")

## 🎯 Próximos Pasos

1. **Revisar sugerencias**: Query gold_cluster_suggestions para ver qué categorías sugirió el agente
2. **Validación manual**: Aprobar/corregir sugerencias por cluster
3. **Actualizar gold_productos_categorias**: Insertar productos de clusters aprobados con método='agente'
4. **Iterar**: Procesar clusters restantes (clusters 15+)
5. **Training**: Con más datos etiquetados, entrenar modelo supervisado